#Paquetes necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

Modelos preentrenados, visualizando con las utilidades de ultralytics

In [2]:
# Carga del modelo
#model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolo11n-seg.pt') #Máscaras
model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model(filename, show=True)

cv2.destroyAllWindows()


WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 135.0ms
video 1/1 (frame 2/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 17.5ms
video 1/1 (frame 3/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 11.6ms
video 1/1 (frame 4/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 11.7m

Desde cámara, detección con yolo11, modelo nano. Visualización propia con OpenCV

In [3]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]


# Captura desde la webcam
vid = cv2.VideoCapture(0)
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Detecta en la imagen
        results = model(img, stream=True)
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()


0: 480x640 (no detections), 110.5ms
Speed: 1.7ms preprocess, 110.5ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 13.8ms
Speed: 2.2ms preprocess, 13.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 18.8ms
Speed: 2.4ms preprocess, 18.8ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 16.9ms
Speed: 1.7ms preprocess, 16.9ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 13.1ms
Speed: 1.9ms preprocess, 13.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.3ms
Speed: 2.1ms preprocess, 11.3ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 15.5ms
Speed: 1.7ms preprocess, 15.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 10.0ms
Speed: 1.6ms preprocess, 10.0ms

Seguimiento. Requiere instalar lap con pip install lap

In [4]:
from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "plate"]


# Captura desde la webcam
vid = cv2.VideoCapture(0)
track_history = defaultdict(lambda: [])
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])

        if 0:
            if results is not None:
                print(results[0])
                boxes = results[0].boxes.xywh.cpu()
                track_ids = results[0].boxes.id.int().cpu().tolist()
                annotated_frame = results[0].plot()
                for box, track_id in zip(boxes, track_ids):
                    x, y, w, h = box
                    track = track_history[track_id]
                    track.append((float(x), float(y)))
                    if len(track) > 30:
                        track.pop(0)
                    points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)
                cv2.imshow("YOLO11 Tracking", annotated_frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
        

        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Destruye ventanas
cv2.destroyAllWindows()




0: 480x640 (no detections), 14.3ms
Speed: 2.3ms preprocess, 14.3ms inference, 3.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.8ms
Speed: 1.7ms preprocess, 11.8ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.5ms
Speed: 1.5ms preprocess, 11.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 12.8ms
Speed: 2.2ms preprocess, 12.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 12.7ms
Speed: 1.8ms preprocess, 12.7ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.9ms
Speed: 1.5ms preprocess, 9.9ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.5ms
Speed: 1.5ms preprocess, 9.5ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 8.5ms
Speed: 1.6ms preprocess, 8.5ms inferen

Intregración con seguimiento (tracking)
!!!!!!!!!Nota: he tenido que bajar a la versión de python 3.9.5 e instalar lap con pip install lap

In [4]:
# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolov11n-seg.pt') #Máscaras
#model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model.track(source=filename, show=True)  # BoT-SORT tracker (por defecto)
#results = model.track(source=filename, show=True, tracker="bytetrack.yaml")  # ByteTrack tracker

cv2.destroyAllWindows()


WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 59.5ms
video 1/1 (frame 2/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 55.1ms
video 1/1 (frame 3/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 53.3ms
video 1/1 (frame 4/375) c:\Users\luisp\Desktop\VC\prac1\P4\TGC23_PdH_C0056cut.mp4: 384x640 3 persons, 52.2ms
video 1/1 (frame 5/375)

In [4]:

data_path = r"C:\Users\luisp\Desktop\VC\prac1\P4\matriculas\data\matriculas.yaml"
model = YOLO("yolo11n.pt")

results = model.train(
    data=data_path,
    epochs=120,             # más largo + patience
    patience=30,
    imgsz=1280,             # sube resolución (mejora detalle)
    batch=-1,               # AutoBatch
    device=0,
    amp=True,
    rect=True,              # mantiene aspect ratio (mejor para cajas finas)
    workers=4,

    optimizer="adamw",
    cos_lr=True,
    lr0=8e-4,               # un poco más bajo para afinar
    lrf=1e-2,

    # Aumentos centrados en “placa pequeña / vídeo”
    mosaic=0.25,            # menos agresivo → mejores cajas
    close_mosaic=15,
    mixup=0.0,              # fuera (deforma letras)
    copy_paste=0.0,
    fliplr=0.0,             # no inviertas texto
    degrees=3.0,            # leve tilt
    translate=0.05,
    scale=0.50,             # importante: genera placas muy pequeñas y grandes
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,  # cambios de luz contrastes

    # Guardados y nombre
    save_period=10,
    project="runs/detect",
    name="plates_s_1280_rect",
    exist_ok=True,
)


Ultralytics 8.3.223  Python-3.9.24 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\luisp\Desktop\VC\prac1\P4\matriculas\data\matriculas.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.25, multi_scale=False, name=plates_s_1280_rect, nbs=64, nms=False, opset=None, optimize=False, optimizer=adamw,

In [5]:
import os, csv, cv2
from collections import defaultdict
from ultralytics import YOLO

# ---------- RUTAS ----------
VIDEO_IN  = r"C:\Users\luisp\Desktop\VC\prac1\P4\videos\video_matriculas.mp4"
VIDEO_OUT = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\video_annotado.mp4"
CSV_OUT   = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\detecciones.csv"

# ---------- MODELOS ----------
detector = YOLO("yolo11n.pt")  # personas/vehículos
plate_model = YOLO(r"C:\Users\luisp\Desktop\VC\prac1\P4\runs\detect\plates_s_1280_rect\weights\best.pt")  # matrículas

# ---------- OPCIONES ----------
TARGET_CLASSES = {"person", "car", "motorbike", "bus", "truck"}
TRACKER = "bytetrack.yaml"     # o BoT-SORT por defecto
DET_CONF = 0.25                # detector general
PLATE_CONF = 0.35              # detector matrículas (ajusta 0.30–0.50)
PLATE_IOU = 0.50
PLATE_IMGSZ = 1280            # 640–1280 según VRAM/recall
PLATE_ONLY_BOTTOM_BAND = True
BOTTOM_FRAC = 0.40             # 40% inferior del vehículo
EXTRA_BAND_UP = 0.05           # +5% hacia arriba por seguridad
MIN_PLATE_AREA = 700           # píxeles
PLATE_AR_MIN, PLATE_AR_MAX = 2.0, 5.0

os.makedirs(os.path.dirname(VIDEO_OUT), exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)

# ---------- VIDEO IO ----------
cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise FileNotFoundError(f"No puedo abrir el vídeo: {VIDEO_IN}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (w, h))

# ---------- CSV ----------
csv_f = open(CSV_OUT, "w", newline="", encoding="utf-8")
cw = csv.writer(csv_f)
cw.writerow([
    "frame","tipo_objeto","confianza","id_tracking","x1","y1","x2","y2",
    "matricula_flag","conf_matricula","mx1","my1","mx2","my2","texto_matricula"
])

# ---------- LOOP ----------
seen_ids = defaultdict(set)
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Tracking del detector general (1 resultado por frame)
    gen = detector.track(
        source=frame, stream=True, persist=True,
        tracker=TRACKER, conf=DET_CONF, verbose=False
    )
    try:
        res = next(gen)
    except StopIteration:
        res = None

    if res is None or res.boxes is None or len(res.boxes) == 0:
        writer.write(frame)
        frame_idx += 1
        continue

    names = detector.model.names
    boxes = res.boxes

    for b in boxes:
        cls_id = int(b.cls[0].item())
        conf   = float(b.conf[0].item())
        name   = names.get(cls_id, str(cls_id))

        if name not in TARGET_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        track_id = int(b.id[0].item()) if b.id is not None else -1
        seen_ids[name].add(track_id)

        color = (0, 255, 0) if name != "person" else (0, 200, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{name} {conf:.2f} ID:{track_id}",
                    (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # -------- MATRÍCULAS SOLO EN VEHÍCULOS (con franja inferior) --------
        plate_flag, plate_conf, plate_box, plate_text = 0, 0.0, (0,0,0,0), ""

        if name in {"car","motorbike","bus","truck"}:
            vx1, vy1, vx2, vy2 = x1, y1, x2, y2

            # ROI = franja inferior del vehículo (+5% hacia arriba)
            if PLATE_ONLY_BOTTOM_BAND:
                vh = vy2 - vy1
                band_top = vy2 - int(BOTTOM_FRAC * vh)
                band_top = band_top - int(EXTRA_BAND_UP * vh)
                rx1, ry1, rx2, ry2 = vx1, band_top, vx2, vy2
            else:
                rx1, ry1, rx2, ry2 = vx1, vy1, vx2, vy2

            # Limitar ROI a los bordes del frame
            rx1 = max(0, rx1); ry1 = max(0, ry1)
            rx2 = min(w, rx2); ry2 = min(h, ry2)

            if rx2 > rx1 and ry2 > ry1:
                crop = frame[ry1:ry2, rx1:rx2]

                p = plate_model.predict(
                    source=crop, conf=PLATE_CONF, iou=PLATE_IOU,
                    imgsz=PLATE_IMGSZ, max_det=3, verbose=False
                )

                if p and len(p[0].boxes) > 0:
                    pb = max(p[0].boxes, key=lambda bb: float(bb.conf[0].item()))
                    px1, py1, px2, py2 = map(int, pb.xyxy[0].tolist())
                    pconf = float(pb.conf[0].item())

                    # —— Reproyección CORRECTA a coords del frame (usar rx1/ry1) ——
                    mx1 = rx1 + px1
                    my1 = ry1 + py1
                    mx2 = rx1 + px2
                    my2 = ry1 + py2

                    # Filtros geométricos
                    wpl, hpl = mx2 - mx1, my2 - my1
                    ar = wpl / max(1, hpl)
                    if (wpl * hpl) >= MIN_PLATE_AREA and PLATE_AR_MIN <= ar <= PLATE_AR_MAX:
                        plate_flag, plate_conf, plate_box = 1, pconf, (mx1, my1, mx2, my2)
                        cv2.rectangle(frame, (mx1, my1), (mx2, my2), (255, 0, 0), 2)
                        cv2.putText(frame, f"PLATE {pconf:.2f}",
                                    (mx1, max(0, my1 - 6)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        mx1, my1, mx2, my2 = plate_box
        cw.writerow([
            frame_idx, name, f"{conf:.3f}", track_id, x1, y1, x2, y2,
            plate_flag, f"{plate_conf:.3f}", mx1, my1, mx2, my2, plate_text
        ])

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
csv_f.close()

print("\nConteo de IDs únicos (aprox):")
for k, ids in seen_ids.items():
    ids.discard(-1)
    print(f"  {k}: {len(ids)}")
print(f"\nVídeo anotado: {VIDEO_OUT}")
print(f"CSV: {CSV_OUT}")



Conteo de IDs únicos (aprox):
  car: 319
  truck: 151
  bus: 40
  person: 8

Vídeo anotado: C:\Users\luisp\Desktop\VC\prac1\P4\outputs\video_annotado.mp4
CSV: C:\Users\luisp\Desktop\VC\prac1\P4\outputs\detecciones.csv
